# Assignment 2: Classification & Regression Analysis

**Course:** Machine Learning S1-2026  
**Total Marks:** 50  
**Due Date:** 9 May 2026, 11:59 PM  

---

## Student Information

**Student Name:** ________________________  
**Student ID:** __________________________  

---

## Assignment Overview

In this assignment you will build an end-to-end machine learning pipeline on a single educational dataset that supports BOTH a regression task (predicting the continuous 2nd-semester grade, 0-20) and a multi-class classification task (predicting whether a student Dropouts, stays Enrolled, or Graduates). You will perform shared preprocessing, then train five regression models and six classification models, run two targeted optimisation tasks, and answer specific questions that require numbers from your own results.

### Submission Format

Submit a single Jupyter Notebook (.ipynb) file containing all code, results, and written analysis. Include your name and student ID. The notebook should run end-to-end without errors.

### Dataset: UCI Predict Students' Dropout and Academic Success

**Load:** Direct zip download from the UCI ML Repository (see starter code in Part 1). A `ucimlrepo` fallback is provided.

- **Samples:** 4,424 students
- **Features:** 36 (demographics, socio-economic factors, academic performance)
- **Regression target:** `Curricular units 2nd sem (grade)` (continuous, 0-20)
- **Classification target:** `Target` (Dropout / Enrolled / Graduate)
- **Known data issues to handle:** header whitespace and a BOM on the first column name; several nominal features are integer-encoded and need one-hot encoding; a subset of rows have a 2nd-sem grade of exactly 0 (students who did not take evaluations); features sit on very different scales; the classification target is mildly imbalanced.

### Instructions
1. Complete all TODO sections following the structured approach.
2. Show all code, outputs, and interpretations.
3. Use the last 2 digits of your student ID as random seed for reproducibility.
4. Provide analysis at each step.
5. Submit the completed notebook with all cells executed.

### Academic Integrity
This assignment requires original analysis. While you may research concepts, all code, analysis, and conclusions must be your own work.

## Setup and Imports

Configure the environment for regression and multi-class classification.

In [ ]:
# Essential imports for the analysis
import pandas as pd
import numpy as np
import io, zipfile, urllib.request, warnings
warnings.filterwarnings('ignore')

# Regression models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBClassifier

# Shared utilities
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from scipy.stats import uniform, randint

# Optional plotting library (not required by any task; kept for reference)
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 6)

# TODO: Replace XX with the last 2 digits of YOUR student ID
STUDENT_SEED = XX  # Example: if your ID ends in 73, use 73; if 00, use 1
np.random.seed(STUDENT_SEED)

---

# Part 1: Loading & Preprocessing (6 Marks)

Load the dataset, identify the two targets, handle data-type issues, encode categorical features, and produce two aligned train/test splits.

In [ ]:
# TODO: Complete the following 6 sub-steps.

# 1. Load the dataset. Primary path: direct UCI zip download (no login needed).
#    Fallback: ucimlrepo. Both are already available in the mlcourse env.

URL = ("https://archive.ics.uci.edu/static/public/697/"
       "predict+students+dropout+and+academic+success.zip")
try:
    with urllib.request.urlopen(URL) as response:
        with zipfile.ZipFile(io.BytesIO(response.read())) as zf:
            with zf.open('data.csv') as f:
                df = pd.read_csv(f, sep=';')
    df.columns = df.columns.str.strip()                         # strip trailing tabs
    df = df.rename(columns={df.columns[0]: 'Marital status'})   # strip BOM
    print(f"Loaded via UCI zip URL: {df.shape}")
except Exception as e:
    print(f"URL load failed ({e}); falling back to ucimlrepo.")
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=697)
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    df.columns = df.columns.str.strip()
    print(f"Loaded via ucimlrepo: {df.shape}")


# 2. Basic inspection. Print df.shape, df.dtypes.value_counts(), the first
#    3 rows, df['Target'].value_counts(), and the describe() of the regression
#    target. Report the EXACT COUNT of rows where the 2nd-sem grade equals 0
#    (students who did not take any evaluations in the 2nd semester - keep
#    them; they are realistic data).

# Your code here:


# 3. Separate the two targets from the feature matrix. Drop BOTH target columns
#    AND 'Curricular units 1st sem (grade)' from X (the 1st-sem grade would
#    leak directly into the 2nd-sem regression target).
#       y_reg = df['Curricular units 2nd sem (grade)']
#       y_cls = df['Target']

# Your code here:


# 4. One-hot encode the nominal integer columns below with
#    pd.get_dummies(drop_first=True). Leave all other columns untouched.
#    Print the resulting X.shape and confirm the column count grew.
#    (After this step, the columns NOT in NOMINAL_COLS are your continuous
#    / binary features - you will need that distinction in Part 2 Model 2.)
NOMINAL_COLS = [
    'Marital status', 'Application mode', 'Course', 'Previous qualification',
    'Nacionality', "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]

# Your code here:


# 5. Create TWO splits on the same X, both with test_size=0.2 and
#    random_state=STUDENT_SEED. Use stratify=y_cls for the classification
#    split only.
#       X_reg_train, X_reg_test, y_reg_train, y_reg_test = ...
#       X_cls_train, X_cls_test, y_cls_train, y_cls_test = ...
#    NOTE: because only the classification split is stratified, the two
#    splits contain DIFFERENT rows. Do not assume X_cls_train and y_reg_train
#    are row-aligned (you will need this fact again in Part 5 Q4).

# Your code here:


# 6. Fit ONE StandardScaler on X_reg_train and ANOTHER on X_cls_train
#    (fit on train only, transform both train and test). Produce the scaled
#    numpy arrays and use exactly these variable names:
#         X_reg_train_s, X_reg_test_s, X_cls_train_s, X_cls_test_s
#    Then encode y_cls_train and y_cls_test with a LabelEncoder, producing:
#         y_cls_train_enc, y_cls_test_enc
#    and keep `le` + `le.classes_` for later cells.

# Your code here:

---

# Part 2: Regression Modelling (15 Marks)

Predict the continuous 2nd-semester grade with five regressors. For each model, train it, evaluate it (MSE, RMSE, MAE, R2), and answer a couple of short questions about what the model did.

In [ ]:
# Part 2 - Regression Modelling
#
# For each of the 5 models below:
#   (a) TRAIN     - fit the model on X_reg_train_s, y_reg_train using the
#                   hyperparameters given.
#   (b) EVALUATE  - predict on X_reg_test_s and print MSE, RMSE, MAE, R2.
#   (c) QUESTIONS - short written answers in the comments.
#
# Carry the numbers into the Part 5 Regression table.


# --- Model 1: Linear Regression ---

# (a) Train a LinearRegression model on the scaled regression training set.

# Your code here:


# (b) Evaluate on the test set: MSE, RMSE, MAE, R2.

# Your code here:


# Q1. Looking at the fitted coefficients (model.coef_), name the TWO features
#     with the largest positive coefficients and the TWO with the largest
#     negative coefficients. Quote their values.
# Q2. What is the intercept (model.intercept_)? Interpret it in the context
#     of predicting a 2nd-semester grade.

# Your answers:



# --- Model 2: Polynomial Regression ---

# (a) Train polynomial regression at degree 2 AND degree 3. Expand ONLY the
#     continuous columns - leave the one-hot dummies unchanged (expanding
#     the dummies would explode the feature count).

# Your code here:


# (b) Evaluate BOTH degrees on the test set.

# Your code here:


# Q1. Which degree generalises better on the TEST set? Quote both R2 values.
# Q2. After degree-3 expansion, how many features does the linear stage see?
#     Why is that a warning sign about overfitting?

# Your answers:



# --- Model 3: SVR (RBF kernel) ---

# (a) Train SVR(kernel='rbf', C=1.0, gamma='scale', epsilon=0.1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Explain what a SMALLER gamma would do to the RBF
#     kernel's decision surface (think about how far each training point
#     "reaches").
# Q2. How many support vectors does your SVR use (len(model.support_))?
#     As a fraction of the training set, is the model using MOST of the
#     data or a SPARSE subset?

# Your answers:



# --- Model 4: Random Forest Regressor ---

# (a) Train RandomForestRegressor(n_estimators=300, max_depth=None,
#                                 min_samples_split=2,
#                                 random_state=STUDENT_SEED, n_jobs=-1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Print the top-5 features by feature_importances_ as a pandas Series
#     (name -> importance).
# Q2. What does max_depth=None do to the trees in the forest? What is the actual depth of your trained trees?
# Q3. What is the trade-off of using very deep trees in a Random Forest?

# Your answers:



# --- Model 5: LightGBM Regressor ---

# (a) Train LGBMRegressor(n_estimators=500, learning_rate=0.1, num_leaves=31,
#                         random_state=STUDENT_SEED, n_jobs=-1, verbose=-1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Print the top-5 features by GAIN importance as a pandas Series:
#         model.booster_.feature_importance(importance_type='gain')
#     Are they the same top-5 as your Random Forest Regressor above?
# Q2. Describe what INCREASING num_leaves does to model
#     capacity, and the trade-off that comes with it.

# Your answers:


---

# Part 3: Classification Modelling (15 Marks)

Predict Dropout / Enrolled / Graduate with six classifiers. For each model, train it with the given hyperparameters, evaluate it (accuracy, macro precision / recall / F1, full `classification_report`), and answer a couple of short questions about what the model did.

In [ ]:
# Part 3 - Classification Modelling
#
# For each of the 6 models below:
#   (a) TRAIN     - fit the model on X_cls_train_s, y_cls_train_enc using
#                   the hyperparameters given.
#   (b) EVALUATE  - predict on X_cls_test_s; print accuracy, macro
#                   precision / recall / F1, and the full classification_report
#                   (use target_names=class_names).
#   (c) QUESTIONS - short written answers in the comments.
#
# Carry the numbers into the Part 5 Classification table.

class_names = list(le.classes_)   # ['Dropout', 'Enrolled', 'Graduate']


# --- Model 1: Logistic Regression ---

# (a) Train LogisticRegression(C=1.0, multi_class='multinomial',
#                              solver='lbfgs', max_iter=2000,
#                              random_state=STUDENT_SEED).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. In Logistic Regression, what does a SMALL value of C mean about the
#     strength of regularisation, and how does that affect the size of the
#     fitted coefficients?
# Q2. In your classification_report, which class has the LOWEST recall and
#     which has the HIGHEST recall? What does that tell you about which
#     outcome the model is under-detecting?

# Your answers:



# --- Model 2: SVC (RBF) ---

# (a) Train SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
#               random_state=STUDENT_SEED).
#     Note: probability=True is required for Task 4.2.

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Describe what a LARGER C would do to the decision
#     boundary of an SVC (think about how much the model is allowed to
#     misclassify training points).
# Q2. Compare the SVC macro-F1 to the Logistic Regression macro-F1. Which
#     wins, and by how much?

# Your answers:



# --- Model 3: Decision Tree ---

# (a) Train DecisionTreeClassifier(max_depth=15, min_samples_split=2,
#                                  criterion='gini',
#                                  random_state=STUDENT_SEED).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Print the tree's depth (model.get_depth()) and total node count
#     (model.tree_.node_count).
# Q2. State what the GINI impurity and the ENTROPY
#     criterion each measure, and how they guide a split.

# Your answers:



# --- Model 4: Random Forest ---

# (a) Train RandomForestClassifier(n_estimators=300, max_depth=None,
#                                  min_samples_split=2,
#                                  class_weight='balanced',
#                                  random_state=STUDENT_SEED, n_jobs=-1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Print the top-5 features by feature_importances_ as a pandas Series.
# Q2. Why does class_weight='balanced' matter here given that Graduate is
#     ~50%, Dropout ~32%, and Enrolled only ~18%? Answer in one sentence.

# Your answers:



# --- Model 5: XGBoost ---

# (a) Train XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6,
#                         objective='multi:softprob', num_class=3,
#                         eval_metric='mlogloss',
#                         random_state=STUDENT_SEED, n_jobs=-1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Print the top-5 features by GAIN importance as a pandas Series:
#         model.get_booster().get_score(importance_type='gain')
#     Are they the same top-5 as the Random Forest classifier?
# Q2. XGBoost usually uses SHALLOW trees (max_depth 3-8) while a single
#     Decision Tree often grows much deeper. Explain in one sentence why
#     shallow is enough when you have hundreds of boosted trees.

# Your answers:



# --- Model 6: LightGBM ---

# (a) Train LGBMClassifier(n_estimators=500, learning_rate=0.1, num_leaves=31,
#                          objective='multiclass', class_weight='balanced',
#                          random_state=STUDENT_SEED, n_jobs=-1, verbose=-1).

# Your code here:


# (b) Evaluate on the test set.

# Your code here:


# Q1. Compare LightGBM's macro-F1 to XGBoost's macro-F1. Which wins, and by
#     how much?
# Q2. LightGBM controls tree shape via num_leaves instead of max_depth. In
#     one sentence, explain why num_leaves can produce very asymmetric
#     (leaf-wise) trees.

# Your answers:



# --- Confusion matrices for your TOP-2 classifiers by macro-F1 ---
# Pick the two classifiers above with the highest macro-F1. For each:
#     cm    = confusion_matrix(y_cls_test_enc, y_pred)
#     cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
#     print(name); print(cm_df)
# No heatmap needed.

# Your code here:


---

# Part 4: Optimisation Tasks (8 Marks)

Two concrete optimisation challenges built on top of your best Part-3 classifier.

In [ ]:
# Task 4.1 - Hyperparameter optimisation with RandomizedSearchCV (4 marks)
#
# Out of Random Forest, XGBoost, or LightGBM, take whichever classifier from Part 3 had the highest macro-F1. Now tune its hyperparameters with RandomizedSearchCV and see
# how much macro-F1 you can pick up. Below are distribution-based search
# spaces for each of the three families - use the one that matches YOUR
# Part-3 winner.

# If your Part-3 winner was LightGBM:
#   est  = LGBMClassifier(objective='multiclass', class_weight='balanced',
#                         random_state=STUDENT_SEED, n_jobs=-1, verbose=-1)
#   dist = {'n_estimators':      randint(200, 800),
#           'learning_rate':     uniform(0.02, 0.15),
#           'num_leaves':        randint(15, 127),
#           'min_child_samples': randint(5, 40),
#           'reg_alpha':         uniform(0, 1),
#           'reg_lambda':        uniform(0, 1)}

# If your Part-3 winner was XGBoost:
#   est  = XGBClassifier(objective='multi:softprob', num_class=3,
#                        eval_metric='mlogloss',
#                        random_state=STUDENT_SEED, n_jobs=-1)
#   dist = {'n_estimators':     randint(200, 800),
#           'learning_rate':    uniform(0.02, 0.15),
#           'max_depth':        randint(3, 10),
#           'subsample':        uniform(0.6, 0.4),
#           'colsample_bytree': uniform(0.6, 0.4),
#           'reg_lambda':       uniform(0, 2)}

# If your Part-3 winner was Random Forest:
#   est  = RandomForestClassifier(random_state=STUDENT_SEED, n_jobs=-1,
#                                 class_weight='balanced')
#   dist = {'n_estimators':      randint(200, 800),
#           'max_depth':         randint(5, 30),
#           'min_samples_split': randint(2, 20),
#           'min_samples_leaf':  randint(1, 10),
#           'max_features':      uniform(0.3, 0.7)}


# (a) Run RandomizedSearchCV(est, dist, n_iter=30, cv=3, scoring='f1_macro',
#     random_state=STUDENT_SEED, n_jobs=-1) on X_cls_train_s, y_cls_train_enc.

# Your code here:


# (b) Evaluate the tuned best_estimator_ on the test set and compute macro-F1.

# Your code here:


# (c) Print three numbers:
#       - the Part-3 baseline macro-F1 for this family
#       - the tuned macro-F1 (test set)
#       - the absolute improvement (tuned - baseline)

# Your code here:


# Q1. Why is sampling 30 random hyperparameter combinations often more
#     efficient than exhaustively trying every combination on a dense grid?
#     Answer in one sentence.
# Q2. Did the randomised search actually beat the baseline? Answer strictly
#     from the numbers you printed above.

# Your answers:


In [ ]:
# Task 4.2 - Dropout-recall optimisation via threshold tuning (4 marks)
#
# A student-support team cares most about catching Dropouts EARLY. Missing a
# Dropout (false negative) is far worse than flagging an Enrolled student
# for extra help. Tune the decision threshold of your best classifier to
# MAXIMISE Dropout recall, SUBJECT TO Dropout precision >= 0.60.

DROPOUT_IDX = class_names.index('Dropout')   # label-encoded index for 'Dropout'


# (a) Take your best classifier from Part 3 (must expose predict_proba).
#     Call it best_cls. Then compute the predicted probability of the
#     Dropout class on the test set:
#         proba         = best_cls.predict_proba(X_cls_test_s)
#         proba_dropout = proba[:, DROPOUT_IDX]

# Your code here:


# (b) Sweep thresholds t in np.arange(0.10, 0.91, 0.02). At each t:
#       - If proba_dropout >= t  -> predict Dropout (class index DROPOUT_IDX).
#       - Otherwise              -> predict the argmax over the NON-Dropout
#                                   probability columns.
#     Compute binary Dropout precision and recall where the positive class
#     is "predicted as Dropout" and the ground truth is
#     "y_cls_test_enc == DROPOUT_IDX". Record each (threshold, precision,
#     recall) in a DataFrame or list.

# Your code here:


# (c) Choose the threshold that MAXIMISES recall subject to
#     precision >= 0.60. If no threshold meets the precision floor, print a
#     message and fall back to the threshold with the highest F1 for Dropout.

# Your code here:


# (d) Also compute Dropout precision and recall under the DEFAULT argmax
#     prediction (best_cls.predict(X_cls_test_s)). Then print:
#       - chosen threshold
#       - Dropout precision and recall at that threshold
#       - Dropout precision and recall at the default argmax prediction
#       - number of ADDITIONAL Dropout students correctly caught by the tuned
#         threshold vs the default
#       - number of Enrolled and Graduate students NOW MIS-LABELLED as Dropout
#         under the tuned threshold (needed for Part 5 Q5)

# Your code here:


# Q1. What does the default argmax prediction correspond to, and why is it
#     NOT the same as using a fixed threshold of 0.5 in a multi-class setting?
# Q2. At your chosen threshold, what is the Dropout recall and what is the
#     Dropout precision? In one sentence, describe the precision-recall
#     trade-off you made relative to the default argmax.

# Your answers:


---

# Part 5: Results Tables & Targeted Analysis (6 Marks)

## Performance Summary

Fill the tables below using the numbers you printed in Parts 2, 3, and 4.

### Regression Results

| Model | R2 | RMSE | MAE | MSE |
|-------|----|------|-----|-----|
| Linear Regression | | | | |
| Polynomial (degree=2) | | | | |
| Polynomial (degree=3) | | | | |
| SVR (best) | | | | |
| Random Forest (best) | | | | |
| LightGBM (best) | | | | |

### Classification Results

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 | Dropout F1 | Enrolled F1 | Graduate F1 |
|-------|----------|-----------------|--------------|----------|------------|-------------|-------------|
| Logistic Regression | | | | | | | |
| SVC (best) | | | | | | | |
| Decision Tree (best) | | | | | | | |
| Random Forest (best) | | | | | | | |
| XGBoost (best) | | | | | | | |
| LightGBM (best) | | | | | | | |

### Optimisation Results

| Task | Baseline | Optimised | Improvement |
|------|----------|-----------|-------------|
| 4.1 RandomizedSearch - macro-F1 | | | |
| 4.2 Threshold tuning - Dropout recall at precision >= 0.60 | | | |

## Targeted Analysis

Answer each question with the exact numbers from your tables above. One or two sentences per answer is enough - no long essays.

**Q1. Per-class winner.** From your classification table, which model has the highest F1 for the **Dropout** class, which for the **Enrolled** class, and which for the **Graduate** class? Quote each F1. Are they the same model for all three classes?

[Your answer]

**Q2. Simple vs complex (regression).** Quote the R2 of Linear Regression and the R2 of your best tree-based regressor (Random Forest or LightGBM). How much R2 did the added complexity buy you? Was the jump worth it?

[Your answer]

**Q3. Feature overlap across tasks.** List the features that appear in BOTH the Random Forest Regressor top-5 and the Random Forest Classifier top-5. For each overlapping feature quote its regression importance and its classification importance side by side.

[Your answer]

**Q4. Regression as a classifier.** Your regression and classification splits contain DIFFERENT rows (only the classification split is stratified), so you cannot directly reuse the model you already fitted in Part 2. Instead:

1. Build a classification-aligned version of the regression target by re-running `train_test_split` while stratifying on `y_cls` but passing `y_reg` as the target:
   ```python
   _, _, y_reg_cls_train, y_reg_cls_test = train_test_split(
       X, y_reg, test_size=0.2, random_state=STUDENT_SEED, stratify=y_cls)
   ```
   Now `X_cls_train_s` and `y_reg_cls_train` are row-aligned.
2. Fit your best regression model type (Linear, SVR, Random Forest, or LightGBM - whichever gave the best R2 in Part 2) on `X_cls_train_s` / `y_reg_cls_train`.
3. Predict continuous grades on `X_cls_test_s` and convert to classes with the rule: `grade < 5 -> Dropout, 5 <= grade < 10 -> Enrolled, grade >= 10 -> Graduate`.
4. Compute accuracy and macro-F1 of the converted predictions against `y_cls_test` (not `y_reg_cls_test`).

Report the rule-based accuracy and macro-F1 next to your best DIRECT classifier's accuracy and macro-F1. By how many percentage points does the direct classifier beat the rule-based conversion on each metric?

[Your answer]

**Q5. Cost of the Task 4.2 threshold.** At the Dropout threshold you chose in Task 4.2, how many Enrolled students and how many Graduate students are now mis-labelled as Dropout? Quote the two counts (already printed in Task 4.2 part d). In one sentence, state whether this false-positive cost is acceptable for a student-support team that uses this model to allocate counselling slots.

[Your answer]

---

# Submission Guidelines

1. Submit a single Python file or Jupyter notebook (`.ipynb`).
2. **File Name Format:**
   `StudentID_Name_Assignment2.ipynb` **or** `StudentID_Name_Assignment2.py`
3. Include all explanatory answers as **comments in the code**.
4. Ensure coding answers and plots appear **inline** (if using a notebook) or **saved as files** (if using `.py`).
5. Maintain **readable and well-commented code**.

> The `.ipynb` or `.py` file should be uploaded into Moodle through the **Assignment 2 Submission** module (under Assessments, *"Assignment 2 Submission Link"*) before the due date.

---

## Instructions for Artificial Intelligence Use

You **may** use Generative AI tools (e.g., ChatGPT, Bard, Grammarly Go) to:

* Clarify concepts, theories, and ideas discussed in class during preparation.
* Generate preliminary ideas for writing and coding.
* Edit a working draft of the assessment.

You **may not** use Generative AI to:

* Generate definitions or writing used in your **final submission**.
* Produce a **paragraph or section of text** for submission.
* Copy and paste **comments or code** directly into the assignment.

**Important:** Using Generative AI for assessable work may be treated as **third-party assistance** (similar to asking someone else to complete your work). Submitting AI-generated material as your own **without acknowledgement** may be considered **plagiarism** and academic misconduct.

---

## Late Submission of Assignments

Assignments submitted late **without an APC (Affected Performance Consideration)** will be penalised:

* **Within 24 hours of the deadline:** 10% deducted
* **After 24 hours and up to 48 hours:** 20% deducted
* **Later than 48 hours:** No grade will be awarded

> Assignments more than 48 hours late will **not be marked** unless APC applies. It is always better to submit an **incomplete assignment on time**.

---

## Affected Performance Consideration (APC)

If circumstances beyond your control affect your ability to complete an assignment, test, or exam, you should complete the **APC form** available at Student Central or online:
[https://www.unitec.ac.nz/current-students/study-support/affected-performance-consideration](https://www.unitec.ac.nz/current-students/study-support/affected-performance-consideration)

---

## Assistance to Other Students

Helping peers is valuable, but only certain types of assistance are acceptable.

### Beneficial Assistance

* Study groups
* Class discussions
* Sharing reading material
* Testing another student's program using **executable code** and sharing results

### Unacceptable Assistance

* Working together on **one copy** of the assessment and submitting it as individual work
* Giving another student your work
* Copying another person's work (including from outside the course)
* Editing or correcting another student's work
* Copying from books, websites, or the Internet and submitting it as your own

---

## Need Help?

To improve your learning and grades, you can:

* Talk with your lecturer
* Visit **Student Success and Achievement** for support
* Visit the **Centre for Pacific Development and Support**
* Visit the **Centre for Māori Development and Support**